# March Machine Learning Mania 2026
## Strong Forecasting Notebook (Standalone, No Public-LB Tricks)

This notebook implements a full, ready forecasting pipeline for both men's and women's tournaments, focused on **generalizable predictive performance** rather than leaderboard leakage.

Will this approach hold on and achieve a better performance than the normal tricks for 0.00 LB or not ? We'l see soon enough.

### Core Principles
- Use only pre-tournament information pathways in a season-aware framework.
- Build rich team-strength features from detailed regular-season statistics.
- Combine diverse model families and calibrate probabilities for Brier score quality.
- Keep the workflow reproducible and publication-friendly.


## Method Summary

### 1) Team-Season Feature Stack
- Possession-based offensive/defensive efficiency features
- Recency-weighted form indicators
- Strength-of-schedule adjustments
- Elo ratings with margin and location effects
- SRS-style opponent-adjusted ratings
- Men's Massey ordinal aggregates
- Conference-level strength context
- Seed and model-implied seed context

### 2) Matchup Construction
For each tournament matchup, team features are merged for Team1 and Team2, then transformed into:
- directional differences
- absolute differences
- selected additive context features

### 3) Modeling and Ensembling
- Elastic-net logistic regression
- LightGBM
- XGBoost
- CatBoost
- Elo-derived heuristic probability

OOF predictions are blended with Brier-optimized weights, then calibrated via logistic calibration.


## Validation Design

Validation is **rolling by season**:
- Train on seasons `< S`
- Validate on season `S`

This is stricter than random CV and better aligned with how future tournament forecasting actually works.


## Configuration

The notebook auto-detects Kaggle and local paths.

- For true 2026 forecasting, prefer `SampleSubmissionStage2.csv`.
- If your current competition phase requires Stage1 formatting, switch to `SampleSubmissionStage1.csv`.


In [1]:
from pathlib import Path
import json
import pandas as pd

# Optional manual override; leave as None to auto-detect data directory.
DATA_DIR_OVERRIDE = None

# Choose the required sample template explicitly for reproducibility.
# Recommended for true 2026 forecasting: SampleSubmissionStage2.csv
SAMPLE_FILE = 'SampleSubmissionStage1.csv'

OUTPUT_FILE = 'submission.csv'
METRICS_FILE = 'cv_metrics.json'
N_VAL_SEASONS = 10


## Pipeline Code

The next cell contains the full standalone implementation.


In [2]:
"""
Strong pipeline for Kaggle March Machine Learning Mania 2026.

Highlights:
- Unified men + women modeling.
- Team-season features from regular-season detailed box scores.
- Elo + SRS ratings.
- Men's final-day Massey ordinal aggregates.
- Conference-strength features.
- Season-aware CV (rolling holdout by season).
- Ensemble of Logistic, LightGBM, XGBoost, CatBoost + Elo heuristic.
- Brier-optimized blending and logistic calibration on OOF predictions.
"""

from __future__ import annotations

import argparse
import json
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier, early_stopping
from scipy.optimize import minimize
from scipy.special import logit
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


EPS = 1e-6
GLOBAL_RANDOM_SEED = 2026
warnings.filterwarnings(
    "ignore",
    message="X does not have valid feature names, but LGBMClassifier was fitted with feature names",
)


@dataclass
class PipelineResult:
    output_path: Path
    metrics_path: Path
    cv_brier: float
    ensemble_weights: Dict[str, float]
    target_season: int
    sample_file: str


def safe_div(numerator: np.ndarray | pd.Series, denominator: np.ndarray | pd.Series) -> np.ndarray:
    num = np.asarray(numerator, dtype=float)
    den = np.asarray(denominator, dtype=float)
    return np.divide(num, den, out=np.zeros_like(num, dtype=float), where=np.abs(den) > EPS)


def parse_seed_num(seed_series: pd.Series) -> pd.Series:
    return seed_series.astype(str).str[1:3].astype(float)


def infer_data_dir(explicit_data_dir: Optional[Path]) -> Path:
    candidates = []
    if explicit_data_dir is not None:
        candidates.append(explicit_data_dir)
    candidates.extend(
        [
            Path("/kaggle/input/competitions/march-machine-learning-mania-2026"),
            Path("/kaggle/input/march-machine-learning-mania-2026"),
            Path("/kaggle/input/march-machine-learning-mania-2026/dataset"),
        ]
    )
    for c in candidates:
        if c.exists() and (c / "MRegularSeasonDetailedResults.csv").exists():
            return c
    raise FileNotFoundError("Could not locate competition data directory with expected CSV files.")


def infer_sample_file(data_dir: Path, explicit_sample_file: Optional[str]) -> str:
    if explicit_sample_file:
        candidate = data_dir / explicit_sample_file
        if not candidate.exists():
            raise FileNotFoundError(f"Sample file not found: {candidate}")
        return explicit_sample_file

    # Default to Stage1 because that is usually required before 2026 games begin.
    for name in ["SampleSubmissionStage1.csv", "SampleSubmissionStage2.csv"]:
        if (data_dir / name).exists():
            return name
    raise FileNotFoundError("No sample submission file found in data directory.")


def read_csv(data_dir: Path, file_name: str, usecols: Optional[Sequence[str]] = None) -> pd.DataFrame:
    return pd.read_csv(data_dir / file_name, usecols=usecols)


def build_team_game_rows(detailed: pd.DataFrame, gender_flag: int) -> pd.DataFrame:
    loc_flip = {"H": "A", "A": "H", "N": "N"}

    wins = pd.DataFrame(
        {
            "Season": detailed["Season"].to_numpy(),
            "DayNum": detailed["DayNum"].to_numpy(),
            "TeamID": detailed["WTeamID"].to_numpy(),
            "OppTeamID": detailed["LTeamID"].to_numpy(),
            "Loc": detailed["WLoc"].to_numpy(),
            "Win": np.ones(len(detailed), dtype=np.float32),
            "PF": detailed["WScore"].to_numpy(dtype=np.float32),
            "PA": detailed["LScore"].to_numpy(dtype=np.float32),
            "FGM": detailed["WFGM"].to_numpy(dtype=np.float32),
            "FGA": detailed["WFGA"].to_numpy(dtype=np.float32),
            "FGM3": detailed["WFGM3"].to_numpy(dtype=np.float32),
            "FGA3": detailed["WFGA3"].to_numpy(dtype=np.float32),
            "FTM": detailed["WFTM"].to_numpy(dtype=np.float32),
            "FTA": detailed["WFTA"].to_numpy(dtype=np.float32),
            "OR": detailed["WOR"].to_numpy(dtype=np.float32),
            "DR": detailed["WDR"].to_numpy(dtype=np.float32),
            "Ast": detailed["WAst"].to_numpy(dtype=np.float32),
            "TO": detailed["WTO"].to_numpy(dtype=np.float32),
            "Stl": detailed["WStl"].to_numpy(dtype=np.float32),
            "Blk": detailed["WBlk"].to_numpy(dtype=np.float32),
            "PFoul": detailed["WPF"].to_numpy(dtype=np.float32),
            "OFGM": detailed["LFGM"].to_numpy(dtype=np.float32),
            "OFGA": detailed["LFGA"].to_numpy(dtype=np.float32),
            "OFGM3": detailed["LFGM3"].to_numpy(dtype=np.float32),
            "OFGA3": detailed["LFGA3"].to_numpy(dtype=np.float32),
            "OFTM": detailed["LFTM"].to_numpy(dtype=np.float32),
            "OFTA": detailed["LFTA"].to_numpy(dtype=np.float32),
            "OOR": detailed["LOR"].to_numpy(dtype=np.float32),
            "ODR": detailed["LDR"].to_numpy(dtype=np.float32),
            "OAst": detailed["LAst"].to_numpy(dtype=np.float32),
            "OTO": detailed["LTO"].to_numpy(dtype=np.float32),
            "OStl": detailed["LStl"].to_numpy(dtype=np.float32),
            "OBlk": detailed["LBlk"].to_numpy(dtype=np.float32),
            "OPFoul": detailed["LPF"].to_numpy(dtype=np.float32),
            "NumOT": detailed["NumOT"].to_numpy(dtype=np.float32),
            "GenderFlag": np.full(len(detailed), gender_flag, dtype=np.int8),
        }
    )

    losses = pd.DataFrame(
        {
            "Season": detailed["Season"].to_numpy(),
            "DayNum": detailed["DayNum"].to_numpy(),
            "TeamID": detailed["LTeamID"].to_numpy(),
            "OppTeamID": detailed["WTeamID"].to_numpy(),
            "Loc": detailed["WLoc"].map(loc_flip).fillna("N").to_numpy(),
            "Win": np.zeros(len(detailed), dtype=np.float32),
            "PF": detailed["LScore"].to_numpy(dtype=np.float32),
            "PA": detailed["WScore"].to_numpy(dtype=np.float32),
            "FGM": detailed["LFGM"].to_numpy(dtype=np.float32),
            "FGA": detailed["LFGA"].to_numpy(dtype=np.float32),
            "FGM3": detailed["LFGM3"].to_numpy(dtype=np.float32),
            "FGA3": detailed["LFGA3"].to_numpy(dtype=np.float32),
            "FTM": detailed["LFTM"].to_numpy(dtype=np.float32),
            "FTA": detailed["LFTA"].to_numpy(dtype=np.float32),
            "OR": detailed["LOR"].to_numpy(dtype=np.float32),
            "DR": detailed["LDR"].to_numpy(dtype=np.float32),
            "Ast": detailed["LAst"].to_numpy(dtype=np.float32),
            "TO": detailed["LTO"].to_numpy(dtype=np.float32),
            "Stl": detailed["LStl"].to_numpy(dtype=np.float32),
            "Blk": detailed["LBlk"].to_numpy(dtype=np.float32),
            "PFoul": detailed["LPF"].to_numpy(dtype=np.float32),
            "OFGM": detailed["WFGM"].to_numpy(dtype=np.float32),
            "OFGA": detailed["WFGA"].to_numpy(dtype=np.float32),
            "OFGM3": detailed["WFGM3"].to_numpy(dtype=np.float32),
            "OFGA3": detailed["WFGA3"].to_numpy(dtype=np.float32),
            "OFTM": detailed["WFTM"].to_numpy(dtype=np.float32),
            "OFTA": detailed["WFTA"].to_numpy(dtype=np.float32),
            "OOR": detailed["WOR"].to_numpy(dtype=np.float32),
            "ODR": detailed["WDR"].to_numpy(dtype=np.float32),
            "OAst": detailed["WAst"].to_numpy(dtype=np.float32),
            "OTO": detailed["WTO"].to_numpy(dtype=np.float32),
            "OStl": detailed["WStl"].to_numpy(dtype=np.float32),
            "OBlk": detailed["WBlk"].to_numpy(dtype=np.float32),
            "OPFoul": detailed["WPF"].to_numpy(dtype=np.float32),
            "NumOT": detailed["NumOT"].to_numpy(dtype=np.float32),
            "GenderFlag": np.full(len(detailed), gender_flag, dtype=np.int8),
        }
    )

    games = pd.concat([wins, losses], ignore_index=True)
    games["Margin"] = games["PF"] - games["PA"]
    games["Poss"] = games["FGA"] - games["OR"] + games["TO"] + 0.475 * games["FTA"]
    games["OppPoss"] = games["OFGA"] - games["OOR"] + games["OTO"] + 0.475 * games["OFTA"]
    games["Pace"] = 0.5 * (games["Poss"] + games["OppPoss"])

    games["OffRtg"] = 100.0 * safe_div(games["PF"], games["Poss"])
    games["DefRtg"] = 100.0 * safe_div(games["PA"], games["OppPoss"])
    games["NetRtg"] = games["OffRtg"] - games["DefRtg"]

    games["eFG"] = safe_div(games["FGM"] + 0.5 * games["FGM3"], games["FGA"])
    games["Opp_eFG"] = safe_div(games["OFGM"] + 0.5 * games["OFGM3"], games["OFGA"])
    games["TOVPct"] = safe_div(games["TO"], games["Poss"])
    games["Opp_TOVPct"] = safe_div(games["OTO"], games["OppPoss"])
    games["ORBPct"] = safe_div(games["OR"], games["OR"] + games["ODR"])
    games["Opp_ORBPct"] = safe_div(games["OOR"], games["OOR"] + games["DR"])
    games["FTR"] = safe_div(games["FTM"], games["FGA"])
    games["Opp_FTR"] = safe_div(games["OFTM"], games["OFGA"])

    games["IsHome"] = (games["Loc"] == "H").astype(np.float32)
    games["IsAway"] = (games["Loc"] == "A").astype(np.float32)
    games["IsNeutral"] = (games["Loc"] == "N").astype(np.float32)
    games["CloseGame"] = (games["Margin"].abs() <= 5).astype(np.float32)
    games["CloseWin"] = games["CloseGame"] * games["Win"]
    games["RecencyW"] = np.exp((games["DayNum"].to_numpy(dtype=np.float32) - 132.0) / 30.0)
    return games


def aggregate_team_features(games: pd.DataFrame) -> pd.DataFrame:
    id_cols = ["Season", "TeamID"]
    group = games.groupby(id_cols, as_index=False)

    mean_cols = [
        "PF",
        "PA",
        "Margin",
        "OffRtg",
        "DefRtg",
        "NetRtg",
        "Pace",
        "eFG",
        "Opp_eFG",
        "TOVPct",
        "Opp_TOVPct",
        "ORBPct",
        "Opp_ORBPct",
        "FTR",
        "Opp_FTR",
        "NumOT",
    ]
    std_cols = ["Margin", "NetRtg", "Pace"]
    weighted_cols = [
        "Margin",
        "OffRtg",
        "DefRtg",
        "NetRtg",
        "Pace",
        "eFG",
        "Opp_eFG",
        "TOVPct",
        "Opp_TOVPct",
        "ORBPct",
        "Opp_ORBPct",
        "FTR",
        "Opp_FTR",
    ]

    base = group.agg(
        Games=("Win", "size"),
        Wins=("Win", "sum"),
        WinPct=("Win", "mean"),
        HomeRate=("IsHome", "mean"),
        AwayRate=("IsAway", "mean"),
        NeutralRate=("IsNeutral", "mean"),
        CloseWins=("CloseWin", "sum"),
        CloseGames=("CloseGame", "sum"),
    )

    mean_df = games.groupby(id_cols, as_index=False)[mean_cols].mean()
    mean_df.columns = [f"{c}_mean" for c in mean_df.columns]
    mean_df = mean_df.rename(columns={"Season_mean": "Season", "TeamID_mean": "TeamID"})

    std_df = games.groupby(id_cols, as_index=False)[std_cols].std().fillna(0.0)
    std_df.columns = [f"{c}_std" for c in std_df.columns]
    std_df = std_df.rename(columns={"Season_std": "Season", "TeamID_std": "TeamID"})

    agg = base.merge(mean_df, on=id_cols, how="left").merge(std_df, on=id_cols, how="left")
    agg["CloseWinPct"] = safe_div(agg["CloseWins"], agg["CloseGames"])

    wr = games[id_cols + ["RecencyW"] + weighted_cols].copy()
    for c in weighted_cols:
        wr[c] = wr[c] * wr["RecencyW"]
    wr = wr.groupby(id_cols, as_index=False).sum()
    for c in weighted_cols:
        wr[f"{c}_wmean"] = safe_div(wr[c], wr["RecencyW"])
    wr = wr[id_cols + [f"{c}_wmean" for c in weighted_cols]]

    recent = (
        games[games["DayNum"] >= 118]
        .groupby(id_cols, as_index=False)[["Win", "Margin", "OffRtg", "DefRtg", "NetRtg", "Pace"]]
        .mean()
    )
    recent.columns = id_cols + [f"{c}_recent" for c in ["Win", "Margin", "OffRtg", "DefRtg", "NetRtg", "Pace"]]

    home_wpct = games[games["Loc"] == "H"].groupby(id_cols, as_index=False)["Win"].mean().rename(
        columns={"Win": "HomeWinPct"}
    )
    away_wpct = games[games["Loc"] == "A"].groupby(id_cols, as_index=False)["Win"].mean().rename(
        columns={"Win": "AwayWinPct"}
    )
    neut_wpct = games[games["Loc"] == "N"].groupby(id_cols, as_index=False)["Win"].mean().rename(
        columns={"Win": "NeutralWinPct"}
    )

    agg = (
        agg.merge(wr, on=id_cols, how="left")
        .merge(recent, on=id_cols, how="left")
        .merge(home_wpct, on=id_cols, how="left")
        .merge(away_wpct, on=id_cols, how="left")
        .merge(neut_wpct, on=id_cols, how="left")
    )

    for c in ["HomeWinPct", "AwayWinPct", "NeutralWinPct"]:
        agg[c] = agg[c].fillna(agg["WinPct"])

    return agg


def add_schedule_features(games: pd.DataFrame, team_feats: pd.DataFrame) -> pd.DataFrame:
    id_cols = ["Season", "TeamID"]
    base = team_feats[id_cols + ["WinPct", "NetRtg_wmean"]].rename(
        columns={"TeamID": "OppTeamID", "WinPct": "OppWinPct_base", "NetRtg_wmean": "OppNet_base"}
    )
    opp = games[["Season", "TeamID", "OppTeamID", "RecencyW"]].merge(base, on=["Season", "OppTeamID"], how="left")
    opp["OppWinPct_base"] = opp["OppWinPct_base"].fillna(0.5)
    opp["OppNet_base"] = opp["OppNet_base"].fillna(0.0)
    opp["w_opp_win"] = opp["OppWinPct_base"] * opp["RecencyW"]
    opp["w_opp_net"] = opp["OppNet_base"] * opp["RecencyW"]

    grp = opp.groupby(["Season", "TeamID"], as_index=False).agg(
        OppWeight=("RecencyW", "sum"),
        OppWinW=("w_opp_win", "sum"),
        OppNetW=("w_opp_net", "sum"),
    )
    grp["SOS_WinPct"] = safe_div(grp["OppWinW"], grp["OppWeight"])
    grp["SOS_NetRtg"] = safe_div(grp["OppNetW"], grp["OppWeight"])
    grp = grp[["Season", "TeamID", "SOS_WinPct", "SOS_NetRtg"]]

    out = team_feats.merge(grp, on=["Season", "TeamID"], how="left")
    out["SOS_WinPct"] = out["SOS_WinPct"].fillna(0.5)
    out["SOS_NetRtg"] = out["SOS_NetRtg"].fillna(0.0)
    out["AdjNetRtg"] = out["NetRtg_wmean"] - out["SOS_NetRtg"]
    return out


def compute_elo(reg_compact: pd.DataFrame, gender_flag: int, base_k: float = 22.0, hca: float = 80.0) -> pd.DataFrame:
    reg_compact = reg_compact.sort_values(["Season", "DayNum"]).reset_index(drop=True)
    seasons = sorted(reg_compact["Season"].unique().tolist())
    prev_end: Dict[int, float] = {}
    rows: List[Tuple[int, int, float, int]] = []

    for season in seasons:
        s_df = reg_compact[reg_compact["Season"] == season]
        teams = np.unique(np.concatenate([s_df["WTeamID"].to_numpy(), s_df["LTeamID"].to_numpy()]))
        ratings: Dict[int, float] = {}
        for t in teams:
            prior = prev_end.get(int(t), 1500.0)
            ratings[int(t)] = 0.75 * prior + 0.25 * 1500.0

        for r in s_df.itertuples(index=False):
            w = int(r.WTeamID)
            l = int(r.LTeamID)
            rw = ratings[w]
            rl = ratings[l]

            if r.WLoc == "H":
                rw_adj, rl_adj = rw + hca, rl
            elif r.WLoc == "A":
                rw_adj, rl_adj = rw, rl + hca
            else:
                rw_adj, rl_adj = rw, rl

            ew = 1.0 / (1.0 + (10.0 ** ((rl_adj - rw_adj) / 400.0)))
            margin = float(r.WScore - r.LScore)
            gap = abs(rw - rl)
            mov_mult = ((abs(margin) + 3.0) ** 0.8) / (7.5 + 0.006 * gap)
            k = base_k * mov_mult
            delta = k * (1.0 - ew)
            ratings[w] = rw + delta
            ratings[l] = rl - delta

        for t in teams:
            rows.append((season, int(t), ratings[int(t)], gender_flag))
            prev_end[int(t)] = ratings[int(t)]

    elo = pd.DataFrame(rows, columns=["Season", "TeamID", "Elo", "GenderFlag"])
    elo["Elo"] = elo["Elo"].astype(float)
    elo["EloZ"] = elo.groupby(["Season", "GenderFlag"])["Elo"].transform(
        lambda s: safe_div(s - s.mean(), s.std(ddof=0) + EPS)
    )
    return elo


def compute_srs(games: pd.DataFrame, ridge: float = 0.08) -> pd.DataFrame:
    rows: List[Tuple[int, int, float, Optional[int]]] = []
    use_gender = "GenderFlag" in games.columns
    group_cols = ["Season", "GenderFlag"] if use_gender else ["Season"]

    for key, group_df in games.groupby(group_cols):
        if use_gender:
            season, gender_flag = int(key[0]), int(key[1])
        else:
            season, gender_flag = int(key), None

        s_df = group_df[["TeamID", "OppTeamID", "Margin"]]
        teams = sorted(s_df["TeamID"].unique().tolist())
        n = len(teams)
        if n < 2:
            continue

        team_to_idx = {t: i for i, t in enumerate(teams)}
        gcount = s_df.groupby("TeamID").size().reindex(teams).fillna(0.0).to_numpy(dtype=float)
        mov = s_df.groupby("TeamID")["Margin"].mean().reindex(teams).fillna(0.0).to_numpy(dtype=float)

        M = np.zeros((n, n), dtype=float)
        edge_counts = s_df.groupby(["TeamID", "OppTeamID"]).size().reset_index(name="Cnt")
        for r in edge_counts.itertuples(index=False):
            i = team_to_idx[int(r.TeamID)]
            j = team_to_idx[int(r.OppTeamID)]
            if gcount[i] > 0:
                M[i, j] = float(r.Cnt) / gcount[i]

        A = np.eye(n, dtype=float) - M
        A += ridge * np.eye(n, dtype=float)
        try:
            rating = np.linalg.solve(A, mov)
        except np.linalg.LinAlgError:
            rating = np.linalg.lstsq(A, mov, rcond=None)[0]
        rating = rating - rating.mean()

        for t, val in zip(teams, rating):
            rows.append((season, t, float(val), gender_flag))

    out = pd.DataFrame(rows, columns=["Season", "TeamID", "SRS", "GenderFlag"])
    if not use_gender:
        out = out.drop(columns=["GenderFlag"])
    return out


def build_massey_features(massey: pd.DataFrame) -> pd.DataFrame:
    m = massey.copy()
    m = m.sort_values(["Season", "SystemName", "RankingDayNum"])
    m = m.groupby(["Season", "SystemName", "TeamID"], as_index=False).tail(1)

    sys_coverage = m.groupby("SystemName")["Season"].nunique()
    keep_systems = sys_coverage[sys_coverage >= 8].index.tolist()
    m = m[m["SystemName"].isin(keep_systems)]

    agg = m.groupby(["Season", "TeamID"])["OrdinalRank"].agg(["mean", "median", "std", "min", "max", "count"]).reset_index()
    agg.columns = [
        "Season",
        "TeamID",
        "MasseyRankMean",
        "MasseyRankMedian",
        "MasseyRankStd",
        "MasseyRankBest",
        "MasseyRankWorst",
        "MasseySystemCount",
    ]
    agg["MasseyRankStd"] = agg["MasseyRankStd"].fillna(0.0)
    agg["MasseyStrength"] = -agg["MasseyRankMean"]
    agg["MasseyStrengthZ"] = agg.groupby("Season")["MasseyStrength"].transform(
        lambda s: safe_div(s - s.mean(), s.std(ddof=0) + EPS)
    )
    return agg


def add_seed_features(team_feats: pd.DataFrame, seeds: pd.DataFrame) -> pd.DataFrame:
    s = seeds.copy()
    s["SeedNum"] = parse_seed_num(s["Seed"])
    s["SeedRegion"] = s["Seed"].astype(str).str[0]
    s["SeedPlayIn"] = (s["Seed"].astype(str).str.len() == 4).astype(np.int8)
    s = s[["Season", "TeamID", "SeedNum", "SeedPlayIn"]]
    out = team_feats.merge(s, on=["Season", "TeamID"], how="left")
    out["HasActualSeed"] = out["SeedNum"].notna().astype(np.int8)
    return out


def add_power_and_expected_seed(team_feats: pd.DataFrame) -> pd.DataFrame:
    out = team_feats.copy()
    group_cols = ["Season", "GenderFlag"]
    z_cols = []
    for c in ["AdjNetRtg", "EloZ", "SRS", "WinPct", "NetRtg_recent", "Margin_recent"]:
        if c in out.columns:
            zname = f"{c}_z"
            out[zname] = out.groupby(group_cols)[c].transform(
                lambda s: safe_div(s - s.mean(), s.std(ddof=0) + EPS)
            )
            z_cols.append(zname)

    power = np.zeros(len(out), dtype=float)
    weights = {
        "AdjNetRtg_z": 0.30,
        "EloZ_z": 0.23,
        "SRS_z": 0.20,
        "WinPct_z": 0.12,
        "NetRtg_recent_z": 0.10,
        "Margin_recent_z": 0.05,
    }
    for c, w in weights.items():
        if c in out.columns:
            power += w * out[c].fillna(0.0).to_numpy(dtype=float)
    out["PowerComposite"] = power

    out["PowerRank"] = out.groupby(group_cols)["PowerComposite"].rank(method="average", ascending=False)
    out["ExpectedSeed"] = np.where(
        out["PowerRank"] <= 68.0,
        np.ceil(out["PowerRank"] / 4.0),
        17.0 + (out["PowerRank"] - 68.0) / 8.0,
    )
    out["ExpectedSeed"] = out["ExpectedSeed"].clip(1.0, 30.0)
    out["ModelSeed"] = out["SeedNum"].fillna(out["ExpectedSeed"])
    out["SeedKnown"] = out["SeedNum"].notna().astype(np.int8)
    return out


def add_conference_features(team_feats: pd.DataFrame, team_confs: pd.DataFrame) -> pd.DataFrame:
    merged = team_feats.merge(team_confs[["Season", "TeamID", "ConfAbbrev"]], on=["Season", "TeamID"], how="left")
    conf = (
        merged.groupby(["Season", "GenderFlag", "ConfAbbrev"], as_index=False)
        .agg(
            ConfPowerMean=("PowerComposite", "mean"),
            ConfPowerStd=("PowerComposite", "std"),
            ConfSize=("TeamID", "nunique"),
            ConfWinPct=("WinPct", "mean"),
        )
        .fillna({"ConfPowerStd": 0.0})
    )
    out = merged.merge(conf, on=["Season", "GenderFlag", "ConfAbbrev"], how="left")
    out["ConfPowerMean"] = out["ConfPowerMean"].fillna(0.0)
    out["ConfPowerStd"] = out["ConfPowerStd"].fillna(0.0)
    out["ConfSize"] = out["ConfSize"].fillna(1.0)
    out["ConfWinPct"] = out["ConfWinPct"].fillna(0.5)
    out["ConfCode"] = pd.Categorical(out["ConfAbbrev"].fillna("UNK")).codes
    return out


def build_team_history_fallback(team_feats: pd.DataFrame, feature_cols: Sequence[str]) -> pd.DataFrame:
    x = team_feats.sort_values(["TeamID", "Season"]).copy()
    x["RankInHist"] = x.groupby("TeamID").cumcount(ascending=True)
    x["MaxRank"] = x.groupby("TeamID")["RankInHist"].transform("max")
    x["HistW"] = 1.0 + 0.4 * (x["RankInHist"] - x["MaxRank"] + 4.0).clip(lower=0.0)
    for c in feature_cols:
        x[f"{c}_w"] = x[c] * x["HistW"]
    grp = x.groupby("TeamID", as_index=False)
    out = grp[[f"{c}_w" for c in feature_cols]].sum()
    wsum = grp["HistW"].sum().rename(columns={"HistW": "HistW_sum"})
    out = out.merge(wsum, on="TeamID", how="left")
    for c in feature_cols:
        out[c] = safe_div(out[f"{c}_w"], out["HistW_sum"])
    keep = ["TeamID"] + list(feature_cols)
    return out[keep]


def build_training_matchups(tourney: pd.DataFrame, gender_flag: int) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "Season": tourney["Season"].astype(int),
            "Team1": np.minimum(tourney["WTeamID"], tourney["LTeamID"]).astype(int),
            "Team2": np.maximum(tourney["WTeamID"], tourney["LTeamID"]).astype(int),
            "Target": (tourney["WTeamID"] < tourney["LTeamID"]).astype(np.float32),
            "GenderFlag": np.full(len(tourney), gender_flag, dtype=np.int8),
        }
    )
    return out


def parse_submission_ids(sample_df: pd.DataFrame) -> pd.DataFrame:
    p = sample_df["ID"].astype(str).str.split("_", expand=True)
    out = sample_df.copy()
    out["Season"] = p[0].astype(int)
    out["Team1"] = p[1].astype(int)
    out["Team2"] = p[2].astype(int)
    out["GenderFlag"] = (out["Team1"] >= 3000).astype(np.int8)
    return out


def build_pair_features(
    pairs: pd.DataFrame,
    team_feats: pd.DataFrame,
    feature_cols: Sequence[str],
    history_fallback: Optional[pd.DataFrame] = None,
) -> pd.DataFrame:
    keep_cols = ["Season", "TeamID"] + list(feature_cols) + ["ConfAbbrev"]
    feat = team_feats[keep_cols].copy()

    left = feat.rename(columns={"TeamID": "Team1", "ConfAbbrev": "T1_ConfAbbrev"})
    left = left.rename(columns={c: f"T1_{c}" for c in feature_cols})

    right = feat.rename(columns={"TeamID": "Team2", "ConfAbbrev": "T2_ConfAbbrev"})
    right = right.rename(columns={c: f"T2_{c}" for c in feature_cols})

    out = pairs.merge(left, on=["Season", "Team1"], how="left")
    out = out.merge(right, on=["Season", "Team2"], how="left")

    if history_fallback is not None:
        h1 = history_fallback.rename(columns={"TeamID": "Team1"})
        h1 = h1.rename(columns={c: f"H1_{c}" for c in feature_cols})
        h2 = history_fallback.rename(columns={"TeamID": "Team2"})
        h2 = h2.rename(columns={c: f"H2_{c}" for c in feature_cols})
        out = out.merge(h1, on="Team1", how="left").merge(h2, on="Team2", how="left")
        for c in feature_cols:
            out[f"T1_{c}"] = out[f"T1_{c}"].fillna(out[f"H1_{c}"])
            out[f"T2_{c}"] = out[f"T2_{c}"].fillna(out[f"H2_{c}"])
        drop_cols = [f"H1_{c}" for c in feature_cols] + [f"H2_{c}" for c in feature_cols]
        out = out.drop(columns=drop_cols)

    engineered = {}
    for c in feature_cols:
        d = out[f"T1_{c}"] - out[f"T2_{c}"]
        engineered[f"{c}_diff"] = d
        engineered[f"{c}_absdiff"] = d.abs()

    for c in ["Pace_wmean", "WinPct", "AdjNetRtg", "ModelSeed", "ConfPowerMean"]:
        if f"T1_{c}" in out.columns and f"T2_{c}" in out.columns:
            engineered[f"{c}_sum"] = out[f"T1_{c}"] + out[f"T2_{c}"]

    if "T1_Elo" in out.columns and "T2_Elo" in out.columns:
        elo_diff = out["T1_Elo"] - out["T2_Elo"]
        engineered["EloWinProbNeutral"] = 1.0 / (1.0 + np.power(10.0, -elo_diff / 400.0))
    else:
        engineered["EloWinProbNeutral"] = np.full(len(out), 0.5, dtype=float)

    engineered["SameConference"] = (out["T1_ConfAbbrev"] == out["T2_ConfAbbrev"]).astype(np.int8)
    out = pd.concat([out, pd.DataFrame(engineered, index=out.index)], axis=1)
    out["T1_ConfAbbrev"] = out["T1_ConfAbbrev"].fillna("UNK")
    out["T2_ConfAbbrev"] = out["T2_ConfAbbrev"].fillna("UNK")
    return out


def select_model_features(pair_df: pd.DataFrame) -> List[str]:
    exclude = {"ID", "Target", "T1_ConfAbbrev", "T2_ConfAbbrev"}
    cols = []
    for c in pair_df.columns:
        if c in exclude:
            continue
        if c in {"Season", "Team1", "Team2"}:
            continue
        if pd.api.types.is_numeric_dtype(pair_df[c]):
            cols.append(c)
    return cols


def fit_and_predict_fold(
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    X_valid: pd.DataFrame,
    y_valid: np.ndarray,
    seed: int,
) -> Tuple[Dict[str, np.ndarray], Dict[str, object]]:
    imputer = SimpleImputer(strategy="median")
    Xtr = imputer.fit_transform(X_train)
    Xva = imputer.transform(X_valid)

    pred: Dict[str, np.ndarray] = {}
    fitted: Dict[str, object] = {"imputer": imputer}

    lr_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    penalty="elasticnet",
                    l1_ratio=0.08,
                    C=0.6,
                    solver="saga",
                    max_iter=6000,
                    random_state=seed,
                ),
            ),
        ]
    )
    lr_pipe.fit(Xtr, y_train)
    pred["lr"] = lr_pipe.predict_proba(Xva)[:, 1]
    fitted["lr"] = lr_pipe

    lgbm = LGBMClassifier(
        objective="binary",
        n_estimators=1300,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=2.0,
        random_state=seed + 11,
        n_jobs=-1,
        verbosity=-1,
    )
    lgbm.fit(
        Xtr,
        y_train,
        eval_set=[(Xva, y_valid)],
        eval_metric="binary_logloss",
        callbacks=[early_stopping(100, verbose=False)],
    )
    pred["lgb"] = lgbm.predict_proba(Xva)[:, 1]
    fitted["lgb"] = lgbm

    xgbm = XGBClassifier(
        objective="binary:logistic",
        n_estimators=1400,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=4,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=2.0,
        gamma=0.0,
        eval_metric="logloss",
        random_state=seed + 29,
        tree_method="hist",
        n_jobs=-1,
    )
    xgbm.fit(
        Xtr,
        y_train,
        eval_set=[(Xva, y_valid)],
        verbose=False,
    )
    pred["xgb"] = xgbm.predict_proba(Xva)[:, 1]
    fitted["xgb"] = xgbm

    cat = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=1000,
        learning_rate=0.025,
        depth=6,
        l2_leaf_reg=6.0,
        random_seed=seed + 47,
        verbose=False,
    )
    cat.fit(Xtr, y_train, eval_set=(Xva, y_valid), use_best_model=True)
    pred["cat"] = cat.predict_proba(Xva)[:, 1]
    fitted["cat"] = cat

    if "EloWinProbNeutral" in X_valid.columns:
        pred["elo"] = X_valid["EloWinProbNeutral"].to_numpy(dtype=float)
    else:
        pred["elo"] = np.full(len(X_valid), 0.5, dtype=float)

    return pred, fitted


def fit_full_models(X: pd.DataFrame, y: np.ndarray, seed: int) -> Dict[str, object]:
    imputer = SimpleImputer(strategy="median")
    Xi = imputer.fit_transform(X)
    bundle: Dict[str, object] = {"imputer": imputer}

    lr_pipe = Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    penalty="elasticnet",
                    l1_ratio=0.08,
                    C=0.6,
                    solver="saga",
                    max_iter=6000,
                    random_state=seed,
                ),
            ),
        ]
    )
    lr_pipe.fit(Xi, y)
    bundle["lr"] = lr_pipe

    lgbm = LGBMClassifier(
        objective="binary",
        n_estimators=900,
        learning_rate=0.02,
        num_leaves=31,
        max_depth=-1,
        min_child_samples=20,
        subsample=0.85,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=2.0,
        random_state=seed + 11,
        n_jobs=-1,
        verbosity=-1,
    )
    lgbm.fit(Xi, y)
    bundle["lgb"] = lgbm

    xgbm = XGBClassifier(
        objective="binary:logistic",
        n_estimators=1200,
        learning_rate=0.02,
        max_depth=4,
        min_child_weight=4,
        subsample=0.9,
        colsample_bytree=0.8,
        reg_alpha=0.05,
        reg_lambda=2.0,
        gamma=0.0,
        eval_metric="logloss",
        random_state=seed + 29,
        tree_method="hist",
        n_jobs=-1,
    )
    xgbm.fit(Xi, y, verbose=False)
    bundle["xgb"] = xgbm

    cat = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="Logloss",
        iterations=800,
        learning_rate=0.025,
        depth=6,
        l2_leaf_reg=6.0,
        random_seed=seed + 47,
        verbose=False,
    )
    cat.fit(Xi, y, verbose=False)
    bundle["cat"] = cat
    return bundle


def predict_full_models(bundle: Dict[str, object], X: pd.DataFrame) -> Dict[str, np.ndarray]:
    imputer: SimpleImputer = bundle["imputer"]  # type: ignore[assignment]
    Xi = imputer.transform(X)
    pred: Dict[str, np.ndarray] = {}
    pred["lr"] = bundle["lr"].predict_proba(Xi)[:, 1]  # type: ignore[index]
    pred["lgb"] = bundle["lgb"].predict_proba(Xi)[:, 1]  # type: ignore[index]
    pred["xgb"] = bundle["xgb"].predict_proba(Xi)[:, 1]  # type: ignore[index]
    pred["cat"] = bundle["cat"].predict_proba(Xi)[:, 1]  # type: ignore[index]
    if "EloWinProbNeutral" in X.columns:
        pred["elo"] = X["EloWinProbNeutral"].to_numpy(dtype=float)
    else:
        pred["elo"] = np.full(len(X), 0.5, dtype=float)
    return pred


def optimize_ensemble_weights(pred_df: pd.DataFrame, y: np.ndarray) -> Dict[str, float]:
    names = list(pred_df.columns)
    p = pred_df.to_numpy(dtype=float)
    n = p.shape[1]

    def obj(w: np.ndarray) -> float:
        z = np.clip(p @ w, 0.001, 0.999)
        return float(np.mean((y - z) ** 2))

    x0 = np.full(n, 1.0 / n, dtype=float)
    bounds = [(0.0, 1.0)] * n
    cons = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]
    res = minimize(obj, x0=x0, method="SLSQP", bounds=bounds, constraints=cons, options={"maxiter": 2000})
    if not res.success:
        w = x0
    else:
        w = res.x
    w = np.clip(w, 0.0, None)
    w = w / max(w.sum(), EPS)
    return {k: float(v) for k, v in zip(names, w)}


def blend_predictions(pred_df: pd.DataFrame, weights: Dict[str, float]) -> np.ndarray:
    z = np.zeros(len(pred_df), dtype=float)
    for k, w in weights.items():
        z += w * pred_df[k].to_numpy(dtype=float)
    return np.clip(z, 0.001, 0.999)


def fit_logit_calibrator(raw_pred: np.ndarray, y: np.ndarray) -> LogisticRegression:
    x = logit(np.clip(raw_pred, 1e-4, 1 - 1e-4)).reshape(-1, 1)
    cal = LogisticRegression(C=50.0, solver="lbfgs", max_iter=2000, random_state=GLOBAL_RANDOM_SEED)
    cal.fit(x, y)
    return cal


def apply_logit_calibrator(cal: LogisticRegression, raw_pred: np.ndarray) -> np.ndarray:
    x = logit(np.clip(raw_pred, 1e-4, 1 - 1e-4)).reshape(-1, 1)
    p = cal.predict_proba(x)[:, 1]
    return np.clip(p, 0.02, 0.98)


def run_pipeline(
    data_dir: Optional[str | Path] = None,
    sample_file: Optional[str] = None,
    output_path: str | Path = "submission.csv",
    metrics_path: str | Path = "cv_metrics.json",
    n_val_seasons: int = 10,
) -> PipelineResult:
    np.random.seed(GLOBAL_RANDOM_SEED)

    data_root = infer_data_dir(Path(data_dir) if data_dir is not None else None)
    sample_name = infer_sample_file(data_root, sample_file)
    sample = read_csv(data_root, sample_name)
    sample_pairs = parse_submission_ids(sample)
    target_season = int(sample_pairs["Season"].mode().iloc[0])

    m_reg_det = read_csv(data_root, "MRegularSeasonDetailedResults.csv")
    w_reg_det = read_csv(data_root, "WRegularSeasonDetailedResults.csv")
    m_reg_comp = read_csv(data_root, "MRegularSeasonCompactResults.csv")
    w_reg_comp = read_csv(data_root, "WRegularSeasonCompactResults.csv")
    m_t = read_csv(data_root, "MNCAATourneyCompactResults.csv")
    w_t = read_csv(data_root, "WNCAATourneyCompactResults.csv")
    m_seeds = read_csv(data_root, "MNCAATourneySeeds.csv")
    w_seeds = read_csv(data_root, "WNCAATourneySeeds.csv")
    m_confs = read_csv(data_root, "MTeamConferences.csv")
    w_confs = read_csv(data_root, "WTeamConferences.csv")
    massey = read_csv(data_root, "MMasseyOrdinals.csv")

    m_reg_det = m_reg_det[m_reg_det["Season"] >= 2003].copy()
    w_reg_det = w_reg_det[w_reg_det["Season"] >= 2010].copy()
    m_reg_comp = m_reg_comp[m_reg_comp["Season"] >= 2003].copy()
    w_reg_comp = w_reg_comp[w_reg_comp["Season"] >= 2010].copy()
    m_t = m_t[m_t["Season"] >= 2003].copy()
    w_t = w_t[w_t["Season"] >= 2010].copy()

    m_games = build_team_game_rows(m_reg_det, gender_flag=0)
    w_games = build_team_game_rows(w_reg_det, gender_flag=1)
    all_games = pd.concat([m_games, w_games], ignore_index=True)

    team_feats = aggregate_team_features(all_games)
    team_feats = add_schedule_features(all_games, team_feats)

    m_elo = compute_elo(m_reg_comp, gender_flag=0)
    w_elo = compute_elo(w_reg_comp, gender_flag=1)
    elo = pd.concat([m_elo, w_elo], ignore_index=True)
    team_feats = team_feats.merge(elo[["Season", "TeamID", "Elo", "EloZ"]], on=["Season", "TeamID"], how="left")

    srs_all = compute_srs(all_games[["Season", "GenderFlag", "TeamID", "OppTeamID", "Margin"]], ridge=0.08)
    srs_recent = compute_srs(
        all_games[all_games["DayNum"] >= 110][["Season", "GenderFlag", "TeamID", "OppTeamID", "Margin"]], ridge=0.12
    ).rename(columns={"SRS": "SRS_recent"})
    if "GenderFlag" in srs_all.columns:
        srs_all = srs_all.drop(columns=["GenderFlag"])
    if "GenderFlag" in srs_recent.columns:
        srs_recent = srs_recent.drop(columns=["GenderFlag"])
    team_feats = team_feats.merge(srs_all, on=["Season", "TeamID"], how="left")
    team_feats = team_feats.merge(srs_recent, on=["Season", "TeamID"], how="left")
    team_feats["SRS_recent"] = team_feats["SRS_recent"].fillna(team_feats["SRS"])

    team_feats["GenderFlag"] = (team_feats["TeamID"] >= 3000).astype(np.int8)

    massey_feats = build_massey_features(massey)
    team_feats = team_feats.merge(massey_feats, on=["Season", "TeamID"], how="left")

    all_seeds = pd.concat([m_seeds, w_seeds], ignore_index=True)
    team_feats = add_seed_features(team_feats, all_seeds)
    team_feats = add_power_and_expected_seed(team_feats)

    all_confs = pd.concat([m_confs, w_confs], ignore_index=True)
    team_feats = add_conference_features(team_feats, all_confs)

    team_numeric_cols = [
        c
        for c in team_feats.columns
        if c not in {"Season", "TeamID", "ConfAbbrev", "SeedNum"} and pd.api.types.is_numeric_dtype(team_feats[c])
    ]

    history_fallback = build_team_history_fallback(team_feats, team_numeric_cols)

    m_train_pairs = build_training_matchups(m_t, gender_flag=0)
    w_train_pairs = build_training_matchups(w_t, gender_flag=1)
    train_pairs = pd.concat([m_train_pairs, w_train_pairs], ignore_index=True)

    train_df = build_pair_features(train_pairs, team_feats, feature_cols=team_numeric_cols, history_fallback=history_fallback)
    train_df = train_df.dropna(subset=["Target"]).reset_index(drop=True)

    feature_cols = select_model_features(train_df)
    X = train_df[feature_cols].copy()
    y = train_df["Target"].to_numpy(dtype=float)

    seasons = sorted(train_df["Season"].unique().tolist())
    n_val_seasons = max(5, min(n_val_seasons, len(seasons) - 2))
    val_seasons = seasons[-n_val_seasons:]

    base_model_names = ["lr", "lgb", "xgb", "cat", "elo"]
    oof_pred = pd.DataFrame(index=train_df.index, columns=base_model_names, dtype=float)
    fold_rows = []

    for i, val_season in enumerate(val_seasons):
        tr_idx = train_df["Season"] < val_season
        va_idx = train_df["Season"] == val_season
        if tr_idx.sum() < 500 or va_idx.sum() == 0:
            continue

        Xtr, ytr = X.loc[tr_idx], y[tr_idx.to_numpy()]
        Xva, yva = X.loc[va_idx], y[va_idx.to_numpy()]

        pred_fold, _ = fit_and_predict_fold(Xtr, ytr, Xva, yva, seed=GLOBAL_RANDOM_SEED + i * 13)
        for k in base_model_names:
            oof_pred.loc[va_idx, k] = np.clip(pred_fold[k], 0.001, 0.999)

        fold_rows.append(
            {
                "season": int(val_season),
                "n_train": int(tr_idx.sum()),
                "n_valid": int(va_idx.sum()),
                "brier_lgb": float(brier_score_loss(yva, np.clip(pred_fold["lgb"], 0.001, 0.999))),
                "brier_cat": float(brier_score_loss(yva, np.clip(pred_fold["cat"], 0.001, 0.999))),
                "brier_xgb": float(brier_score_loss(yva, np.clip(pred_fold["xgb"], 0.001, 0.999))),
            }
        )

    valid_mask = oof_pred.notna().all(axis=1)
    min_required_oof = max(200, int(0.15 * len(train_df)))
    if valid_mask.sum() < min_required_oof:
        raise RuntimeError(
            f"Insufficient OOF rows generated for reliable blending. "
            f"Got {int(valid_mask.sum())}, expected at least {min_required_oof}."
        )

    y_oof = y[valid_mask.to_numpy()]
    pred_oof = oof_pred.loc[valid_mask].copy()

    base_brier = {k: float(brier_score_loss(y_oof, pred_oof[k].to_numpy(dtype=float))) for k in base_model_names}
    weights = optimize_ensemble_weights(pred_oof[base_model_names], y_oof)
    raw_blend_oof = blend_predictions(pred_oof[base_model_names], weights)
    calibrator = fit_logit_calibrator(raw_blend_oof, y_oof)
    cal_oof = apply_logit_calibrator(calibrator, raw_blend_oof)
    cv_brier = float(brier_score_loss(y_oof, cal_oof))

    full_bundle = fit_full_models(X, y, seed=GLOBAL_RANDOM_SEED)

    pred_df = build_pair_features(sample_pairs, team_feats, feature_cols=team_numeric_cols, history_fallback=history_fallback)
    X_test = pred_df[feature_cols].copy()

    base_test = predict_full_models(full_bundle, X_test)
    base_test_df = pd.DataFrame({k: np.clip(v, 0.001, 0.999) for k, v in base_test.items()})
    raw_blend_test = blend_predictions(base_test_df[base_model_names], weights)
    final_pred = apply_logit_calibrator(calibrator, raw_blend_test)

    out = sample[["ID"]].copy()
    out["Pred"] = np.clip(final_pred, 0.02, 0.98)

    out_path = Path(output_path)
    out.to_csv(out_path, index=False)

    metrics = {
        "data_dir": str(data_root),
        "sample_file": sample_name,
        "target_season": target_season,
        "n_train_rows": int(len(train_df)),
        "n_features": int(len(feature_cols)),
        "n_oof_rows": int(valid_mask.sum()),
        "n_submission_rows": int(len(out)),
        "base_brier": base_brier,
        "blend_weights": weights,
        "cv_brier_calibrated": cv_brier,
        "val_seasons": val_seasons,
        "fold_rows": fold_rows,
    }
    metrics_out = Path(metrics_path)
    metrics_out.write_text(json.dumps(metrics, indent=2), encoding="utf-8")

    return PipelineResult(
        output_path=out_path,
        metrics_path=metrics_out,
        cv_brier=cv_brier,
        ensemble_weights=weights,
        target_season=target_season,
        sample_file=sample_name,
    )


## Train and Generate Submission

This executes feature engineering, rolling validation, model fitting, blending, calibration, and submission export.


In [3]:
result = run_pipeline(
    data_dir=DATA_DIR_OVERRIDE,
    sample_file=SAMPLE_FILE,
    output_path=OUTPUT_FILE,
    metrics_path=METRICS_FILE,
    n_val_seasons=N_VAL_SEASONS,
)

result


PipelineResult(output_path=PosixPath('submission.csv'), metrics_path=PosixPath('cv_metrics.json'), cv_brier=0.1642329042529144, ensemble_weights={'lr': 0.00966916530962345, 'lgb': 0.037671749153136445, 'xgb': 0.006723158068984547, 'cat': 0.9135576670506742, 'elo': 0.0323782604175813}, target_season=2025, sample_file='SampleSubmissionStage1.csv')

## Inspect Outputs

Artifacts:
- `submission.csv`
- `cv_metrics.json`


In [4]:
metrics = json.loads(Path(METRICS_FILE).read_text(encoding='utf-8'))
sub = pd.read_csv(OUTPUT_FILE)

print('Submission rows:', len(sub))
print('Pred range:', float(sub.Pred.min()), float(sub.Pred.max()))
print('Output path:', Path(OUTPUT_FILE).resolve())
print('Metrics path:', Path(METRICS_FILE).resolve())

metrics


Submission rows: 519144
Pred range: 0.02 0.98
Output path: /kaggle/working/submission.csv
Metrics path: /kaggle/working/cv_metrics.json


{'data_dir': '/kaggle/input/competitions/march-machine-learning-mania-2026',
 'sample_file': 'SampleSubmissionStage1.csv',
 'target_season': 2025,
 'n_train_rows': 2410,
 'n_features': 344,
 'n_oof_rows': 1315,
 'n_submission_rows': 519144,
 'base_brier': {'lr': 0.1887421355018606,
  'lgb': 0.16934333703268475,
  'xgb': 0.18225085559844917,
  'cat': 0.16415277897203742,
  'elo': 0.18040136768626985},
 'blend_weights': {'lr': 0.00966916530962345,
  'lgb': 0.037671749153136445,
  'xgb': 0.006723158068984547,
  'cat': 0.9135576670506742,
  'elo': 0.0323782604175813},
 'cv_brier_calibrated': 0.1642329042529144,
 'val_seasons': [2015, 2016, 2017, 2018, 2019, 2021, 2022, 2023, 2024, 2025],
 'fold_rows': [{'season': 2015,
   'n_train': 1095,
   'n_valid': 130,
   'brier_lgb': 0.1217673297661085,
   'brier_cat': 0.13415595376098122,
   'brier_xgb': 0.13075661137686231},
  {'season': 2016,
   'n_train': 1225,
   'n_valid': 130,
   'brier_lgb': 0.19202837083122348,
   'brier_cat': 0.176905395563

In [5]:
sub.head()


,ID,Pred
0,2022_1101_1102,0.752986
1,2022_1101_1103,0.374300
2,2022_1101_1104,0.095453
3,2022_1101_1105,0.742026
4,2022_1101_1106,0.829809


## Publishing Notes

If you publish this notebook on Kaggle:
1. Label it as a forecasting-first solution (not a public-LB exploit).
2. Include your runtime settings and validation horizon.
3. Keep `cv_metrics.json` in outputs for reproducibility.
4. Mention that this pipeline is designed to transfer better to true unseen outcomes than historical-override approaches.
